In [4]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    train_path: Path = Path("../data/train.csv")
    test_path: Path = Path("../data/test.csv")
    sub_path: Path = Path("../data/sample_submission.csv")
    pltpd_path: Path = Path("../data/podcast_dataset.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('../data/train.csv'),
 'test_path': PosixPath('../data/test.csv'),
 'sub_path': PosixPath('../data/sample_submission.csv'),
 'pltpd_path': PosixPath('../data/podcast_dataset.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.02,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [5]:
from IPython.display import display
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    df.loc[df['Episode_Length_minutes']>121.0, 'Episode_Length_minutes'] = 121.0

    df['Host_Guest_Diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']
    df['Host_Guest_Ratio'] = (df['Host_Popularity_percentage'] / df['Guest_Popularity_percentage']).replace([float('inf'), -float('inf')], pd.NA)

    if "Listening_Time_minutes" in df.columns:
        df['Listening_Episode_Diff'] = df['Episode_Length_minutes'] - df['Listening_Time_minutes']
        df['Listening_Episode_Ratio'] = (df['Episode_Length_minutes'] / df['Listening_Time_minutes']).replace([float('inf'), -float('inf')], pd.NA)

    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

df_pltpd = pd.read_csv(cfg.pltpd_path)
df_pltpd = df_pltpd.dropna(subset=['Listening_Time_minutes'])
df_pltpd = df_pltpd.reset_index(drop=True)
df_pltpd.index = df_pltpd.index + 1000000

df_train = pd.concat([df_train, df_pltpd], axis=0)
df_train["id"] = df_train.index

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

display(df_train)
display(df_train.describe())
display(df_train.isna().sum())

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
0,0,NaN,0,74.81,3,21,NaN,0.0,2,31.419980,0,98,NaN,NaN,NaN,NaN
1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.012410,1,26,-9.00,0.881501,31.787590,1.361172
2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.925310,2,16,61.00,7.800446,28.974690,1.644952
3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.278240,3,45,-21.48,0.727065,20.891760,1.451438
4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.610310,4,86,21.39,1.364519,34.899690,1.461573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1047100,33,24.81,9,66.15,0,17,98.63,1.0,1,20.573795,1047100,17,-32.48,0.670688,4.236205,1.205903
1047101,11,92.15,6,89.61,5,21,25.82,2.0,0,76.198459,1047101,9,63.79,3.470565,15.951541,1.209342
1047102,23,112.27,1,26.33,5,21,55.29,0.0,1,107.602135,1047102,24,-28.96,0.476216,4.667865,1.043381
1047103,19,NaN,8,41.47,2,14,33.58,0.0,1,17.220998,1047103,85,7.89,1.234961,NaN,NaN


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Listening_Episode_Diff
count,797105.000000,705317.000000,797105.000000,797105.000000,797105.000000,797105.000000,646356.000000,797104.000000,797105.000000,797105.000000,7.971050e+05,797105.000000,646356.000000,705317.000000
mean,23.540988,64.408705,4.554814,59.877839,3.028731,15.663856,52.095246,1.357792,0.998145,45.444668,4.133258e+05,51.378954,7.627507,18.679341
std,13.911304,32.981409,2.962341,22.889880,2.022848,4.027274,28.483819,1.149681,0.815531,27.140915,2.598146e+05,28.131239,36.152160,13.566281
min,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,1.000000,-80.170000,-115.540000
25%,12.000000,35.670000,2.000000,39.450000,1.000000,14.000000,28.100000,0.000000,0.000000,23.184220,1.992760e+05,28.000000,-18.280000,8.130000
50%,23.000000,63.770000,5.000000,60.060000,3.000000,17.000000,53.350000,1.000000,1.000000,43.392270,3.985520e+05,52.000000,6.640000,15.643750
75%,36.000000,94.000000,7.000000,79.560000,5.000000,21.000000,76.490000,2.000000,2.000000,64.814620,5.978280e+05,75.000000,33.000000,26.683090
max,47.000000,121.000000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,119.970000,1.047104e+06,100.000000,113.550000,103.220440


Podcast_Name                        0
Episode_Length_minutes          91788
Genre                               0
Host_Popularity_percentage          0
Publication_Day                     0
Publication_Time                    0
Guest_Popularity_percentage    150749
Number_of_Ads                       1
Episode_Sentiment                   0
Listening_Time_minutes              0
id                                  0
Episode_Num                         0
Host_Guest_Diff                150749
Host_Guest_Ratio               150752
Listening_Episode_Diff          91788
Listening_Episode_Ratio        100172
dtype: int64

In [23]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']

df_dup = df_train.copy()
df_dup = df_dup.dropna(subset=['Guest_Popularity_percentage'])
df_dup = df_dup[df_dup.duplicated(subset=cols_to_compare, keep=False)]
# df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup = df_dup.sort_values(cols_to_compare)
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341,18.236
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341,18.236
163092,0,93.78,0,68.03,5,10,17.16,1.0,1,71.796010,163092,1,50.87,3.964452,21.983990,1.306201,71.796
348103,0,96.02,0,68.03,5,10,17.16,1.0,1,71.796010,348103,1,50.87,3.964452,24.223990,1.3374,71.796
216180,0,115.56,0,98.62,4,14,2.48,1.0,0,106.422180,216180,2,96.14,39.766129,9.137820,1.085864,106.422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044806,47,66.66,6,38.92,1,14,15.98,1.0,1,47.523106,1044806,98,22.94,2.435544,19.136893,1.402686,47.523
123989,47,102.45,6,42.13,6,17,41.29,0.0,1,89.820570,123989,99,0.84,1.020344,12.629430,1.140607,89.821
548936,47,102.45,6,42.13,0,14,41.29,0.0,0,89.820570,548936,99,0.84,1.020344,12.629430,1.140607,89.821
1035367,47,102.45,6,42.13,6,14,41.29,0.0,1,89.820573,1035367,99,0.84,1.020344,12.629427,1.140607,89.821


In [24]:
x = 110
df_dup.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
667785,6,81,85.31,70.40,60.325660,103.82,1,17,1
354247,6,81,86.70,43.35,43.190680,50.26,3,21,2
1012245,6,81,86.70,43.35,43.190678,50.26,6,21,2
425124,6,81,87.99,93.79,18.306070,20.32,0,14,1
562012,6,81,87.99,93.79,18.306070,25.75,0,14,1
648958,6,81,87.99,93.79,18.306070,22.99,0,14,1
1008155,6,81,87.99,93.79,18.306074,28.90,0,14,1
267691,6,81,94.24,26.97,69.589120,93.46,2,14,0
1043963,6,81,94.24,26.97,69.589123,93.46,2,14,2
30611,6,82,23.58,61.24,46.613370,NaN,4,21,1


In [25]:
grouped = df_train.groupby(cols_to_compare)
result = grouped.filter(lambda x: x['Listening_Time_minutes'].nunique() > 1)
result = result.sort_values(cols_to_compare)
result

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
263468,0,19.30,9,21.51,1,10,96.10,0.0,0,18.236090,263468,1,-74.59,0.223829,1.063910,1.058341,18.236
1043473,0,19.30,0,21.51,1,10,96.10,3.0,0,18.236093,1043473,1,-74.59,0.223829,1.063907,1.058341,18.236
216180,0,115.56,0,98.62,4,14,2.48,1.0,0,106.422180,216180,2,96.14,39.766129,9.137820,1.085864,106.422
468002,0,115.56,0,98.62,5,14,2.48,1.0,0,106.422180,468002,2,96.14,39.766129,9.137820,1.085864,106.422
1012076,0,115.56,0,98.62,0,14,2.48,1.0,0,106.422183,1012076,2,96.14,39.766129,9.137817,1.085864,106.422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044806,47,66.66,6,38.92,1,14,15.98,1.0,1,47.523106,1044806,98,22.94,2.435544,19.136893,1.402686,47.523
123989,47,102.45,6,42.13,6,17,41.29,0.0,1,89.820570,123989,99,0.84,1.020344,12.629430,1.140607,89.821
548936,47,102.45,6,42.13,0,14,41.29,0.0,0,89.820570,548936,99,0.84,1.020344,12.629430,1.140607,89.821
1035367,47,102.45,6,42.13,6,14,41.29,0.0,1,89.820573,1035367,99,0.84,1.020344,12.629427,1.140607,89.821


In [26]:
x = 110
result.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
45888,14,16,76.58,61.12,39.718490,65.04,4,10,0
1039809,14,16,76.58,61.12,39.718487,65.04,4,10,0
551187,14,17,25.90,85.56,94.893640,111.51,0,17,2
1036115,14,17,25.90,85.56,94.893643,99.51,0,17,2
514295,14,17,87.67,1.44,91.919570,105.67,3,10,2
521337,14,17,87.67,1.44,91.919570,108.39,3,10,2
1009659,14,17,87.67,1.44,91.919572,107.47,3,10,2
29255,14,18,51.99,92.06,11.398850,17.46,2,17,2
1034225,14,18,51.99,92.06,11.398854,17.46,2,17,2
47908,14,19,90.64,42.99,21.432860,22.90,4,14,2


In [37]:
df_train['Listening_Time_minutes_rounded'] = df_train['Listening_Time_minutes'].round(1)

grouped = df_train.groupby(cols_to_compare)
unique_counts = grouped['Listening_Time_minutes_rounded'].nunique()

groups_with_diff = unique_counts[unique_counts > 1].index
result = df_train[df_train.set_index(cols_to_compare).index.isin(groups_with_diff)]

result = result.sort_values(cols_to_compare)
result

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
321773,0,NaN,0,42.14,6,14,68.24,2.0,1,30.427760,321773,60,-26.10,0.617526,NaN,NaN,30.4
1013894,0,40.48,0,42.14,6,14,68.24,2.0,1,29.427768,1013894,60,-26.10,0.617526,11.052232,1.375572,29.4
374046,2,103.87,2,81.82,6,14,99.43,1.0,2,102.890000,374046,31,-17.61,0.82289,0.980000,1.009525,102.9
497554,2,106.79,2,81.82,3,10,99.43,1.0,2,105.890000,497554,31,-17.61,0.82289,0.900000,1.008499,105.9
1030540,2,106.79,2,81.82,3,10,99.43,1.0,2,105.890008,1030540,31,-17.61,0.82289,0.899992,1.008499,105.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434552,43,32.29,1,79.86,4,10,30.14,0.0,1,27.779330,434552,75,49.72,2.649635,4.510670,1.162375,27.8
486961,43,46.52,1,79.86,4,10,30.14,2.0,2,26.779330,486961,75,49.72,2.649635,19.740670,1.737161,26.8
689947,46,55.78,9,44.20,4,21,91.22,0.0,2,26.964050,689947,93,-47.02,0.484543,28.815950,2.06868,27.0
710059,46,62.65,9,44.20,0,14,91.22,0.0,2,49.964050,710059,93,-47.02,0.484543,12.685950,1.253902,50.0


In [52]:
x = 0
result.iloc[x*30 : (x+1)*30][["Podcast_Name", "Episode_Num", "Host_Popularity_percentage", "Guest_Popularity_percentage", "Listening_Time_minutes", "Episode_Length_minutes", "Publication_Day", "Publication_Time", "Episode_Sentiment"]]

,Podcast_Name,Episode_Num,Host_Popularity_percentage,Guest_Popularity_percentage,Listening_Time_minutes,Episode_Length_minutes,Publication_Day,Publication_Time,Episode_Sentiment
321773,0,60,42.14,68.24,30.427760,NaN,6,14,1
1013894,0,60,42.14,68.24,29.427768,40.48,6,14,1
374046,2,31,81.82,99.43,102.890000,103.87,6,14,2
497554,2,31,81.82,99.43,105.890000,106.79,3,10,2
1030540,2,31,81.82,99.43,105.890008,106.79,3,10,2
78144,3,28,26.84,47.80,21.990000,23.68,3,17,2
488094,3,28,26.84,47.80,20.990000,21.78,2,10,2
163254,5,80,70.26,92.66,55.372760,67.37,2,14,2
428023,5,80,70.26,92.66,55.372760,71.64,2,14,2
569438,5,80,70.26,92.66,53.372760,71.53,2,14,2


In [ ]:
from sklearn.metrics import mean_squared_error

def calc_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse

calc_rmse([89.693310], [69.530000])

20.163309999999996

In [58]:
# Drop duplicate get first by Podcast_Name and Episode_Num
result_f = result.drop_duplicates(subset=['Podcast_Name', 'Episode_Num'], keep='first')["Listening_Time_minutes"]
result_l = result.drop_duplicates(subset=['Podcast_Name', 'Episode_Num'], keep='last')["Listening_Time_minutes"]
calc_rmse(result_f, result_l)

11.042652053609213

In [54]:
grouped.diff()

,Episode_Length_minutes,Genre,Publication_Day,Publication_Time,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,id,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_rounded
321773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1013894,NaN,0.0,0.0,0.0,0.0,0.0,-0.999992,692121.0,0.0,0.0,NaN,NaN,-1.0
374046,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497554,2.92,0.0,-3.0,-4.0,0.0,0.0,3.000000,123508.0,0.0,0.0,-0.080000,-0.001025,3.0
1030540,0.00,0.0,0.0,0.0,0.0,0.0,0.000008,532986.0,0.0,0.0,-0.000008,-0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
434552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486961,14.23,0.0,0.0,0.0,2.0,1.0,-1.000000,52409.0,0.0,0.0,15.230000,0.574786,-1.0
689947,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
710059,6.87,0.0,-4.0,-7.0,0.0,0.0,23.000000,20112.0,0.0,0.0,-16.130000,-0.814779,23.0
